# Fundamentos de observabilidad en agentes LLM

Este notebook introduce qué significa observar un agente de lenguaje, por qué es más complejo que observar una API tradicional y qué información debemos capturar para entender su comportamiento.

## Objetivos
- Comprender la diferencia entre observabilidad y depuración en agentes LLM.
- Identificar las señales clave: prompts, respuestas, modelo, tokens, latencia, costo, errores y herramientas usadas.
- Construir una traza simple de ejecución usando Python y `pandas`.

## Temas
- Observabilidad en software tradicional: logs, métricas y trazas.
- Por qué un agente LLM es más difícil de observar.
- Diferencias entre: llamada simple, cadena de pasos, agente con herramientas, agente con memoria y agente multi-paso.
- Qué registrar en cada ejecución.

## 1. Observabilidad tradicional vs observabilidad de agentes LLM

En sistemas tradicionales, observamos principalmente:
- `logs` de eventos
- `métricas` de rendimiento (latencia, tasa de errores)
- `trazas` de ejecución entre servicios.

Para un agente LLM, además de esos datos, necesitamos saber:
- qué prompt se envió
- qué modelo se usó
- cuántos tokens se consumieron
- si el agente llamó a una herramienta
- por qué tomó una decisión en cada paso.

In [34]:
!pip install pandas langchain langchain-openai langchain_classic wikipedia LangSmith


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [35]:
import wikipedia
from langchain_openai import ChatOpenAI

# Configurar el idioma de Wikipedia
wikipedia.set_lang("es")

# Configuración del LLM
try:
    llm = ChatOpenAI(
        model="gpt-4o",
        openai_api_base=os.environ.get("GITHUB_BASE_URL"),
        openai_api_key=os.environ.get("GITHUB_TOKEN"),
        temperature=0
    )
    print("✅ LLM de LangChain configurado.")
except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    llm = None

✅ LLM de LangChain configurado.


In [36]:
from langchain_classic.agents import tool, create_openai_tools_agent, AgentExecutor
from langchain_classic import hub
from langsmith import Client

@tool
def get_wikipedia_summary(query: str) -> str:
    """Busca en Wikipedia un tema y devuelve un resumen de 2 frases. Útil para obtener información sobre personas, lugares o conceptos."""
    try:
        return wikipedia.summary(query, sentences=2)
    except Exception as e:
        return f"Ocurrió un error: {e}"

tools = [get_wikipedia_summary]

client = Client()
prompt = client.pull_prompt("hormold/openai-functions-agent",
                            dangerously_pull_public_prompt=True)

agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("✅ Agente y herramientas listos.")

✅ Agente y herramientas listos.


In [37]:
# Importar herramientas de análisis y simulación
import os
import time
import uuid
from pprint import pprint

# Intentar importar pandas con mensaje claro si no está instalado
try:
    import pandas as pd
except ImportError:
    raise ImportError(
        'Este notebook usa pandas. Ejecuta `pip install pandas` antes de continuar.'
    )

from langchain_core.messages import HumanMessage

# Función de ayuda para invocar el LLM real

def call_llm(prompt_text):
    if llm is None:
        raise RuntimeError("El LLM no está configurado. Revisa GITHUB_BASE_URL y GITHUB_TOKEN.")

    def extract_text(response):
        if hasattr(response, "content"):
            return response.content
        if hasattr(response, "text"):
            return response.text
        if hasattr(response, "generations") and response.generations:
            return response.generations[0][0].text
        return str(response)

    try:
        message = HumanMessage(content=prompt_text)
        if hasattr(llm, "predict_messages"):
            response = llm.predict_messages([message])
        elif hasattr(llm, "generate"):
            response = llm.generate([[message]])
        else:
            response = llm([message])

        text = extract_text(response)
        model_name = getattr(llm, "model_name", getattr(llm, "model", "gpt-4o"))
        prompt_tokens = 0
        completion_tokens = 0
        if hasattr(response, "usage") and response.usage is not None:
            prompt_tokens = getattr(response.usage, "prompt_tokens", 0)
            completion_tokens = getattr(response.usage, "completion_tokens", 0)
        return {
            "text": text,
            "model": model_name,
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "latency": 0.0,
            "cost": 0.0,
            "error": None,
        }
    except Exception as e:
        return {
            "text": "",
            "model": getattr(llm, "model_name", getattr(llm, "model", "gpt-4o")),
            "prompt_tokens": 0,
            "completion_tokens": 0,
            "latency": 0.0,
            "cost": 0.0,
            "error": str(e),
        }


In [38]:
class ObservabilityTrace:
    def __init__(self):
        self.events = []

    def record(self, event):
        # Guardar una traza simple con las señales más importantes
        self.events.append(event)

    def to_dataframe(self):
        return pd.DataFrame(self.events)

    def display(self):
        df = self.to_dataframe()
        display(df)

# Agente simple con distintos modos de ejecución
class MiniAgent:
    def __init__(self, llm):
        self.llm = llm
        self.memory = []
        self.model_name = getattr(llm, "model_name", getattr(llm, "model", "gpt-4o"))

    def _record_step(self, trace, step_type, prompt, output, decision, error=None):
        trace.record({
            'step_id': str(uuid.uuid4()),
            'step_type': step_type,
            'prompt': prompt,
            'output': output,
            'model': self.model_name,
            'decision': decision,
            'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
            'error': error,
        })

    def _generate(self, prompt):
        return call_llm(prompt)

    def simple_call(self, user_question, trace):
        # Una llamada directa al modelo, sin pasos intermedios
        result = self._generate(user_question)
        self._record_step(
            trace,
            step_type='llm-simple',
            prompt=user_question,
            output=result['text'],
            decision='uso modelo directo',
            error=result['error'],
        )
        return result

    def chain_of_steps(self, user_question, trace):
        # Un agente que transforma la pregunta y luego genera la respuesta
        analysis_prompt = f'Analiza la pregunta para extraer intención: {user_question}'
        analysis = self._generate(analysis_prompt)
        self._record_step(
            trace,
            step_type='llm-analyze',
            prompt=analysis_prompt,
            output=analysis['text'],
            decision='descomponer pregunta',
            error=analysis['error'],
        )

        final_prompt = f'Con base en el análisis previo, responde: {user_question}'
        final = self._generate(final_prompt)
        self._record_step(
            trace,
            step_type='llm-answer',
            prompt=final_prompt,
            output=final['text'],
            decision='respuesta final',
            error=final['error'],
        )
        return final

    def agent_with_tools(self, user_question, trace):
        # Un agente que decide usar una herramienta o no
        decision_prompt = f'¿Necesito una herramienta para contestar: {user_question}?'
        decision = self._generate(decision_prompt)
        self._record_step(
            trace,
            step_type='llm-decision',
            prompt=decision_prompt,
            output=decision['text'],
            decision='evaluar uso de herramienta',
            error=decision['error'],
        )

        if 'calcula' in decision['text'].lower() or 'suma' in decision['text'].lower():
            tool_output = str(sum(int(n) for n in user_question.split() if n.isdigit()))
            self._record_step(
                trace,
                step_type='tool',
                prompt='Calculadora local',
                output=tool_output,
                decision='usar calculadora',
            )
            answer = f'El resultado de la operación es {tool_output}.'
            self._record_step(
                trace,
                step_type='llm-answer',
                prompt=f'Responde usando el resultado de la calculadora: {tool_output}',
                output=answer,
                decision='formular respuesta con herramienta',
            )
            return {'text': answer, 'model': self.model_name, 'prompt_tokens': 0, 'completion_tokens': 0, 'latency': 0.05, 'cost': 0.0, 'error': None}

        answer = self._generate(user_question)
        self._record_step(
            trace,
            step_type='llm-answer',
            prompt=user_question,
            output=answer['text'],
            decision='respuesta directa sin herramienta',
            error=answer['error'],
        )
        return answer

    def memory_agent(self, user_question, trace):
        # Un agente con memoria que recuerda interacciones anteriores
        history = ' | '.join(self.memory[-2:])
        prompt = f'Historial: {history} Pregunta: {user_question}' if history else user_question
        result = self._generate(prompt)
        self.memory.append(user_question)
        self._record_step(
            trace,
            step_type='llm-memory',
            prompt=prompt,
            output=result['text'],
            decision='usar contexto de memoria',
            error=result['error'],
        )
        return result

    def multi_step_agent(self, user_question, trace):
        # Un agente multi-paso combina análisis, herramienta y respuesta
        step1 = self._generate(f'Analiza y resume la pregunta: {user_question}')
        self._record_step(
            trace,
            step_type='llm-analyze',
            prompt=f'Analiza y resume la pregunta: {user_question}',
            output=step1['text'],
            decision='extraer intención',
            error=step1['error'],
        )

        step2 = self._generate(f'¿Deberíamos usar un dato externo para: {user_question}?')
        self._record_step(
            trace,
            step_type='llm-decision',
            prompt=f'¿Deberíamos usar un dato externo para: {user_question}?',
            output=step2['text'],
            decision='evaluar herramienta externa',
            error=step2['error'],
        )

        if 'sí' in step2['text'].lower() or 'necesito' in step2['text'].lower():
            tool_output = '42'
            self._record_step(
                trace,
                step_type='tool',
                prompt='Consulta a base de conocimiento local',
                output=tool_output,
                decision='consulta externa',
            )
            final_text = f'Usé la herramienta externa y el resultado es {tool_output}.'
        else:
            final = self._generate(user_question)
            final_text = final['text']

        self._record_step(
            trace,
            step_type='llm-answer',
            prompt='Genera la respuesta final',
            output=final_text,
            decision='respuesta multi-paso',
            error=None,
        )
        return {'text': final_text, 'model': self.model_name, 'prompt_tokens': 0, 'completion_tokens': 0, 'latency': 0.0, 'cost': 0.0, 'error': None}


In [39]:
# Crear un agente real y registrar varias ejecuciones
if llm is None:
    raise RuntimeError("El LLM no está configurado. Verifica las credenciales y las variables de entorno.")

agent = MiniAgent(llm)

# Ejecución 1: llamada simple a LLM
trace_simple = ObservabilityTrace()
result_simple = agent.simple_call('¿Cuál es la capital de Francia?', trace_simple)
print('=== Respuesta simple ===')
print(result_simple['text'])
print()
trace_simple.display()

# Ejecución 2: cadena de pasos
trace_chain = ObservabilityTrace()
_ = agent.chain_of_steps('Explícame cómo funciona una batería.', trace_chain)
print('=== Traza de cadena de pasos ===')
trace_chain.display()

# Ejecución 3: agente con herramientas
trace_tool = ObservabilityTrace()
_ = agent.agent_with_tools('Calcula 13 + 29.', trace_tool)
print('=== Traza de agente con herramientas ===')
trace_tool.display()

# Ejecución 4: agente con memoria
trace_memory = ObservabilityTrace()
_ = agent.memory_agent('¿Qué fue lo último que pregunté?', trace_memory)
print('=== Traza de agente con memoria ===')
trace_memory.display()

# Ejecución 5: agente multi-paso
trace_multi = ObservabilityTrace()
_ = agent.multi_step_agent('¿Debo usar una herramienta para responder esta pregunta?', trace_multi)
print('=== Traza de agente multi-paso ===')
trace_multi.display()


=== Respuesta simple ===
La capital de Francia es **París**.



,step_id,step_type,prompt,output,model,decision,timestamp,error
0,fc78c895-d421-400c-bb6e-8ae703b7ff5c,llm-simple,¿Cuál es la capital de Francia?,La capital de Francia es **París**.,gpt-4o,uso modelo directo,2026-05-22 10:00:38,None


=== Traza de cadena de pasos ===


,step_id,step_type,prompt,output,model,decision,timestamp,error
0,37418153-b923-46ff-8e40-7a2be232361d,llm-analyze,Analiza la pregunta para extraer intención: Ex...,"La intención detrás de la pregunta **""Explícam...",gpt-4o,descomponer pregunta,2026-05-22 10:00:40,None
1,9bc746eb-e055-4674-a1cd-059ec514f06f,llm-answer,"Con base en el análisis previo, responde: Expl...",¡Claro! Una batería es un dispositivo que alma...,gpt-4o,respuesta final,2026-05-22 10:00:47,None


=== Traza de agente con herramientas ===


,step_id,step_type,prompt,output,model,decision,timestamp,error
0,1d5a65fc-daba-49da-84f9-2ddf64df4db5,llm-decision,¿Necesito una herramienta para contestar: Calc...,"No, no necesitas una herramienta para calcular...",gpt-4o,evaluar uso de herramienta,2026-05-22 10:00:49,None
1,7cc7a1b2-c31e-4fb6-896d-720367648c5f,tool,Calculadora local,13,gpt-4o,usar calculadora,2026-05-22 10:00:49,None
2,884d1e35-240d-4eaa-86cc-65e96da739bd,llm-answer,Responde usando el resultado de la calculadora...,El resultado de la operación es 13.,gpt-4o,formular respuesta con herramienta,2026-05-22 10:00:49,None


=== Traza de agente con memoria ===


,step_id,step_type,prompt,output,model,decision,timestamp,error
0,5925f4c4-7039-42c6-815f-28140a2f5d09,llm-memory,¿Qué fue lo último que pregunté?,"No tengo memoria de tus interacciones previas,...",gpt-4o,usar contexto de memoria,2026-05-22 10:00:51,None


=== Traza de agente multi-paso ===


,step_id,step_type,prompt,output,model,decision,timestamp,error
0,b715ecf0-97df-403e-9727-e81f404b7ca7,llm-analyze,Analiza y resume la pregunta: ¿Debo usar una h...,La pregunta plantea una reflexión sobre la nec...,gpt-4o,extraer intención,2026-05-22 10:00:52,None
1,11e98e00-ecae-49a1-aef3-fe908b76c7df,llm-decision,¿Deberíamos usar un dato externo para: ¿Debo u...,"Esta pregunta parece ser un poco paradójica, p...",gpt-4o,evaluar herramienta externa,2026-05-22 10:00:56,None
2,bdbab31c-bb67-4041-a7e2-1c8a9bd05a15,tool,Consulta a base de conocimiento local,42,gpt-4o,consulta externa,2026-05-22 10:00:56,None
3,da2041b9-41ea-4b0d-8f61-37b53129c031,llm-answer,Genera la respuesta final,Usé la herramienta externa y el resultado es 42.,gpt-4o,respuesta multi-paso,2026-05-22 10:00:56,None
